# 🧲 GIADA roadmap Task 14 — variazioni Ca-HVA
Un run appaiato sui checkpoint Task11 **congelati**. Variazioni separate di conduttanza massima, potenziale di inversione, gate iniziali e maschera del meccanismo. Il riferimento resta il sistema Ca-HVA+pas a un compartimento, con passo interno 0.025 ms e interfaccia modello 1 ms. Non è una prova sul neurone completo. Il supplemento 11c resta separato.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task14');REPO=WORK/'giada'
assert not WORK.exists(),f'Workspace già presente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_parameter_variation.py').is_file(),'Revisione GIADA senza Task14.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.roadmap_current_contract import verified_task11_checkpoints
from src.giada_teacher.roadmap_parameter_variation import ParameterVariationConfig,evaluate_parameter_matrix,_verify_task12_13
print({'code_revision':REVISION})


## 📁 Prerequisiti piccoli, verificati
Input Kaggle: `giada_roadmap_task11_causal_operator.zip` e `giada_roadmap_task12_13_current_contract.zip`, oppure le rispettive cartelle Dataset estratte. Il primo fornisce i modelli congelati; il secondo prova che corrente analitica e timing nativo sono stati verificati. Non serve il dataset HayFlow multi-GB. Se i Dataset sono montati con nomi diversi, imposta `GIADA_TASK11_ARTIFACT` e `GIADA_TASK12_13_ARTIFACT` a ZIP/cartella.


In [ ]:
INPUT=Path('/kaggle/input')
def find_source(env,needles,marker,verifier):
 override=os.environ.get(env);candidates=[Path(override).expanduser()] if override else []
 if INPUT.is_dir():
  candidates += [p for p in INPUT.rglob('*.zip') if any(n in str(p).lower() for n in needles)]
  candidates += [p.parent for p in INPUT.rglob(marker) if any(n in str(p).lower() for n in needles)]
 for candidate in dict.fromkeys(candidates):
  if not candidate.exists():continue
  try:verifier(candidate);return candidate.resolve()
  except (ValueError,KeyError,FileNotFoundError,OSError):continue
 raise AssertionError(f'{env}: artefatto esatto assente. Aggiungi il Dataset o imposta la variabile al relativo ZIP/cartella.')
TASK11=find_source('GIADA_TASK11_ARTIFACT',('task11','task-11','causal-operator','causal_operator'),'selection_freeze.json',verified_task11_checkpoints)
TASK12_13=find_source('GIADA_TASK12_13_ARTIFACT',('task12-13','task12_13','current-contract','current_contract'),'task12_current_report.json',_verify_task12_13)
print({'task11':str(TASK11),'task12_13':str(TASK12_13)})


## 🧪 Matrice appaiata
Stessi 48 episodi seed 14059 e stessi ingressi pianificati per nove condizioni. La baseline a ECa=120 deve riprodurre esattamente la ricorrenza Task11; in caso contrario il run si ferma. `gbar_half` è entro il supporto originale, `gbar_one_half` può uscire dal supporto. I **twin ECa one-step** partono dallo stesso stato e hanno input numerico identico ma target diversi: diagnosticano un'informazione assente. Le traiettorie ECa complete divergono e non hanno input identici ai tempi successivi. La maschera off visibile porta la conduttanza efficace a zero; la variante off nascosta è solo un controllo negativo. Valutiamo one-step, rollout causale 16ms e cambiamenti appaiati di V/corrente, per `path_full` ed `effect_full`, senza training.


In [ ]:
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task14_parameter_variation')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Usa una sessione pulita.'
OUTPUT.mkdir(parents=True)
print({'device':DEVICE,'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None},flush=True)
config=ParameterVariationConfig();config.validate()
report=evaluate_parameter_matrix(TASK11,TASK12_13,device=DEVICE,config=config)
report['code_revision']=REVISION
(OUTPUT/'task14_parameter_matrix_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
compact={name:{'axis':row['axis'],'teacher_delta_V_rms':round(row['teacher_delta_voltage_vs_baseline_rms_mv'],5),'path_V':round(row['arms']['path_full']['one_step_voltage_rmse_mv'],5),'path_m':round(row['arms']['path_full']['one_step_m_rmse'],6),'path_I':round(row['arms']['path_full']['one_step_current_rmse_ma_cm2'],9),'path_16ms_V':row['arms']['path_full']['recursive_16ms_voltage_rmse_mv']} for name,row in report['results'].items()}
display({'valid':report['valid'],'scenarios':report['scenario_count'],'baseline_identity_max':report['baseline_reference_max_abs_error'],'eca_twins_equal_input':report['eca_one_step_twins_equal_numeric_input_tensor'],'eca_teacher_voltage_difference_rms_mv':report['eca_equal_input_teacher_voltage_difference_rms_mv'],'eca_lower_bound_worst_rmse_mv':report['eca_pair_unavoidable_worst_rmse_lower_bound_mv'],'registered_in_domain_gates':report['registered_in_domain_performance_gates']})
display(compact)
if not report['valid']:print('Gate tecnico non superato: scarica comunque il report diagnostico; non interpretare i gate prestazionali.')


## 📦 Scarica i risultati
ZIP piccolo con report completo. Il download usa il metodo Blob/base64 concordato.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task14_parameter_variation','zip',OUTPUT.parent,OUTPUT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
